# Modelling: forecasting `lcc_change`, and do pollution / disturbance matter?

Two separate goals, answered separately:

- **Part A: Prediction.** Which model forecasts next survey's live coral cover change best? Evaluated with a **rolling origin**: for each year from 2018 to 2025, train on all earlier years and forecast that year, tuning settings on the past years only. The best model is then refitted on all 2013–2025 data.
- **Part B: Effects.** Does last survey's pollution or disturbance cover predict coral change, holding heat stress and other factors fixed? Uses a fixed, ecologically chosen feature set with island-bootstrap confidence intervals. Feature selection for prediction can't answer this: it drops any feature that adds little *prediction* next to the strongest ones, which is not the same as having no effect.

Inputs come from `preprocessing.ipynb` (`data/processed/`).

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.inspection import permutation_importance
from sklearn.linear_model import HuberRegressor, Lasso, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

pd.set_option('display.width', 200)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

DATA_DIR = Path('data/processed')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

TARGET = 'lcc_change'
N_FOLDS = 5          # island-grouped folds, used only in the tuning check (A8)
SEED = 42
ORIGINS = list(range(2018, 2026))   # forecast years

data = pd.read_csv(DATA_DIR / 'features_raw.csv')
pre = joblib.load(DATA_DIR / 'preprocessor.joblib')   # clone() gives an unfitted copy
features = list(pre.feature_names_in_)
N_OUT = len(pre.get_feature_names_out())
print(f'{len(data)} rows, {data["island"].nunique()} islands, {data["survey_year"].min()}-{data["survey_year"].max()}; '
      f'{len(features)} input columns -> {N_OUT} after preprocessing')

# Part A: Prediction

## A1. Evaluation design: rolling origin with nested tuning

For each forecast year t:
1. **Training data:** every survey before t.
2. **Tuning:** each model's settings are chosen by **year-ordered CV within those past years**: validate on each of the most recent past years (up to 8, each with at least 30 earlier rows to train on), training on all years before it. So tuning never sees year t, and it asks the same question as the evaluation: how well do these settings forecast a year they haven't seen? Island-grouped folds would mix years, letting year-wide shocks leak between folds and favour features that track them (see A8).
3. **Forecast:** refit on all past years with the chosen settings, predict year t.

This tests every year from 2018 to 2025 exactly once, including the 2024–2025 drop, and later years become training data for the forecasts after them. Starting at 2018 guarantees at least 91 training rows and that the 2016 heat event is in the past. The overall score is the MAE across all 257 forecast rows.

In [ ]:
pd.DataFrame([{'forecast_year': t,
               'train_years': f'2013-{t - 1}',
               'train_rows': int((data['survey_year'] < t).sum()),
               'train_islands': data.loc[data['survey_year'] < t, 'island'].nunique(),
               'test_rows': int((data['survey_year'] == t).sum()),
               'mean_change_that_year': data.loc[data['survey_year'] == t, TARGET].mean().round(2)}
              for t in ORIGINS]).set_index('forecast_year')

## A2. Candidate models

Every candidate is one `Pipeline` (preprocessing → [feature selection] → model), so the imputer, scaler and feature ranking are refitted on each training set and never see the rows they're scored on.

| Candidate | Tuned settings | Grid size |
|---|---|---|
| Training mean | – | 1 |
| No change (predict 0) | – | 1 |
| Ridge, all 37 features | `alpha` | 13 |
| Top-k + Ridge (`SelectKBest(f_regression)`) | `k` = 1…37 × `alpha`, **one-SE rule** | 481 |
| Lasso | `alpha` | 25 |
| Random forest (300 trees) | `max_depth`, `min_samples_leaf`, `max_features` | 18 |
| AdaBoost (decision-tree base) | base tree depth, `n_estimators`, `learning_rate` | 12 |
| Huber, all 37 features (`HuberRegressor`, ε = 1.35) | `alpha` | 13 |
| Top-k + Huber | `k` × `alpha`, one-SE rule | 481 |

**Scoring:** mean absolute error (MAE). **One-SE rule for k:** take the smallest k whose CV MAE is within one standard error of the best, because with only a few validation years the single best k is mostly noise.

### Two ways to reduce the influence of noisy targets

Each change is the difference of two survey averages, and surveys over only 1–3 sites are much noisier (typical change 14 points vs about 6 for 4+ sites).

- **Weighting by survey reliability** (models marked *weighted*: Ridge, top-k + Ridge, Lasso, random forest, Huber, top-k + Huber). The variance of a change is modelled as `a + b / n_eff`: `a` is real year-to-year variation, `b / n_eff` is survey noise, and `n_eff = 1 / (1/n_sites + 1/prev_n_sites)`. `a` and `b` are fitted on the training years at each origin (regressing squared deviations on `1/n_eff`), and each row's weight is `1 / (a + b / n_eff)`, scaled to average 1. Weights only affect training.
- **Huber loss:** squared error for small residuals, absolute error beyond ε = 1.35 scale units, so a −37.5 point outlier pulls the fit much less than in Ridge.

Everything is still **scored unweighted**, on every forecast row, so results compare directly with the unweighted models.

In [ ]:
SCORING = 'neg_mean_absolute_error'
ALPHAS_RIDGE = np.logspace(-2, 4, 13)
ALPHAS_LASSO = np.logspace(-3, 1, 25)
ALPHAS_HUBER = np.logspace(-3, 3, 13)
K_GRID = list(range(1, N_OUT + 1))
MAX_TUNING_YEARS, MIN_TRAIN_ROWS = 8, 30


def time_cv(rows):
    """Year-ordered splits: validate on year v, train on all earlier years."""
    years = rows['survey_year'].to_numpy()
    valid = [v for v in np.unique(years) if (years < v).sum() >= MIN_TRAIN_ROWS][-MAX_TUNING_YEARS:]
    return [(np.flatnonzero(years < v), np.flatnonzero(years == v)) for v in valid]
BASELINES = ['Training mean', 'No change (0)']


def fold_mae(cv_results):
    """Mean and standard error of the validation MAE across folds, one row per setting."""
    fold_cols = [c for c in cv_results if c.startswith('split') and c.endswith('_test_score')]
    mae = -pd.DataFrame(cv_results)[fold_cols]
    return mae.mean(axis=1), mae.std(axis=1, ddof=1) / np.sqrt(len(fold_cols))


def one_se_smallest_k(cv_results):
    """Refit rule: smallest k within 1 SE of the best CV MAE; best alpha for that k."""
    mean, se = fold_mae(cv_results)
    k = pd.Series(cv_results['param_select__k']).astype(int)
    best = mean.idxmin()
    within = mean <= mean[best] + se[best]
    k_min = k[within].min()
    return int(mean[within & (k == k_min)].idxmin())


def pipe(*steps):
    return Pipeline([('pre', clone(pre)), *steps])


CANDIDATES = {
    'Training mean': (pipe(('model', DummyRegressor())), {}),
    'No change (0)': (pipe(('model', DummyRegressor(strategy='constant', constant=0.0))), {}),
    'Ridge (all features)': (pipe(('model', Ridge())), {'model__alpha': ALPHAS_RIDGE}),
    'Top-k + Ridge': (pipe(('select', SelectKBest(f_regression)), ('model', Ridge())),
                      {'select__k': K_GRID, 'model__alpha': ALPHAS_RIDGE}),
    'Lasso': (pipe(('model', Lasso(max_iter=50_000))), {'model__alpha': ALPHAS_LASSO}),
    'Random forest': (pipe(('model', RandomForestRegressor(n_estimators=300, random_state=SEED))),
                      {'model__max_depth': [3, 5, None], 'model__min_samples_leaf': [5, 10, 20],
                       'model__max_features': [0.33, 1.0]}),
    'AdaBoost': (pipe(('model', AdaBoostRegressor(estimator=DecisionTreeRegressor(), random_state=SEED))),
                 {'model__estimator__max_depth': [2, 3, 4], 'model__n_estimators': [100, 300],
                  'model__learning_rate': [0.03, 0.1]}),
    'Huber': (pipe(('model', HuberRegressor(max_iter=1000))), {'model__alpha': ALPHAS_HUBER}),
    'Top-k + Huber': (pipe(('select', SelectKBest(f_regression)), ('model', HuberRegressor(max_iter=1000))),
                      {'select__k': K_GRID, 'model__alpha': ALPHAS_HUBER}),
}
# Weighted variants: same estimator and grid, trained with survey-reliability weights
WEIGHTED = {'Ridge, weighted': 'Ridge (all features)', 'Top-k + Ridge, weighted': 'Top-k + Ridge',
            'Lasso, weighted': 'Lasso', 'Random forest, weighted': 'Random forest',
            'Huber, weighted': 'Huber', 'Top-k + Huber, weighted': 'Top-k + Huber'}
CANDIDATES.update({w: CANDIDATES[base] for w, base in WEIGHTED.items()})
MODELS = list(CANDIDATES)


def noise_weights(rows):
    """Inverse-variance weights from var(change) = a + b / n_eff, fitted on these rows."""
    inv_n = 1 / rows['n_eff_sites'].to_numpy()
    sq_dev = (rows[TARGET] - rows[TARGET].mean()).to_numpy() ** 2
    a, b = np.linalg.lstsq(np.column_stack([np.ones(len(rows)), inv_n]), sq_dev, rcond=None)[0]
    a, b = max(a, 1e-6), max(b, 0.0)
    w = 1 / (a + b * inv_n)
    return w / w.mean(), a, b


def make_search(name, cv_splits):
    estimator, grid = CANDIDATES[name]
    refit = one_se_smallest_k if name.startswith('Top-k') else True
    return GridSearchCV(clone(estimator), grid, cv=cv_splits, scoring=SCORING, refit=refit, n_jobs=-1)


def fit_search(name, cv_splits, rows, weights):
    fit_params = {'model__sample_weight': weights} if name in WEIGHTED else {}
    return make_search(name, cv_splits).fit(rows[features], rows[TARGET], **fit_params)

## A3. Run the rolling origin

Takes about 40 minutes: 15 models are tuned at each of 8 origins, each on up to 8 validation years.

In [ ]:
pred_frames, chosen, fitted, weight_fits = [], [], {}, []
for t in ORIGINS:
    past = data[data['survey_year'] < t].reset_index(drop=True)
    now = data[data['survey_year'] == t].reset_index(drop=True)
    inner_cv = time_cv(past)
    w_past, a, b = noise_weights(past)
    weight_fits.append({'forecast_year': t, 'a_real_variation': a, 'b_survey_noise': b,
                        'min_weight': w_past.min(), 'max_weight': w_past.max()})

    out = now[['island', 'survey_year', TARGET]].copy()
    for name in MODELS:
        search = fit_search(name, inner_cv, past, w_past)
        out[name] = search.predict(now[features])
        fitted[(t, name)] = search.best_estimator_
        chosen.append({'forecast_year': t, 'model': name,
                       **{k.replace('model__', '').replace('estimator__', 'base_'): v
                          for k, v in search.best_params_.items()}})
    pred_frames.append(out)
    print(f'{t}: trained on {len(past)} rows ({past["island"].nunique()} islands), forecast {len(now)} rows; '
          f'tuned on validation years {[int(past["survey_year"].iloc[v[0]]) for _, v in inner_cv]}')

preds = pd.concat(pred_frames, ignore_index=True)
chosen = pd.DataFrame(chosen)
assert len(preds) == int(data['survey_year'].isin(ORIGINS).sum())
pd.DataFrame(weight_fits).set_index('forecast_year').round(2)

## A4. Results

**Per year:** MAE of each model's forecasts for that year. **Overall:** MAE across all forecast rows, the skill relative to the training mean (1 − MAE / MAE of the mean; positive = better), the number of years a model beats the mean, and the average yearly gain over the mean ± its standard error across the 8 years.

**Out-of-sample R² (`R2_oos`)** = 1 − Σ(actual − forecast)² / Σ(actual − training mean)², pooled over all forecast rows. The reference is the mean of the *past* years at each origin (what you'd actually know when forecasting), not the mean of the forecast year itself, which would need the future. 0 = no better than the training mean; negative = worse. Unlike MAE, it squares errors, so large misses weigh more.

In [ ]:
rows = []
for t, g in preds.groupby('survey_year'):
    rows.append({'forecast_year': t, 'rows': len(g), 'mean_change': g[TARGET].mean(),
                 **{m: mean_absolute_error(g[TARGET], g[m]) for m in MODELS}})
by_year = pd.DataFrame(rows).set_index('forecast_year')

mae_all = pd.Series({m: mean_absolute_error(preds[TARGET], preds[m]) for m in MODELS})
sse = pd.Series({m: ((preds[TARGET] - preds[m]) ** 2).sum() for m in MODELS})
gain = by_year[MODELS].rsub(by_year['Training mean'], axis=0)   # positive = better than the mean
overall = pd.DataFrame({
    'MAE': mae_all,
    'R2_oos': 1 - sse / sse['Training mean'],
    'skill_vs_mean': 1 - mae_all / mae_all['Training mean'],
    'years_beating_mean': (gain > 0).sum(),
    'mean_yearly_gain': gain.mean(),
    'gain_SE': gain.std(ddof=1) / np.sqrt(len(ORIGINS)),
}).sort_values('MAE')

display(by_year.round(2))
overall.round(3)

### Did weighting or Huber help?

Each treated model against its untreated version: change in MAE (negative = better), change in out-of-sample R² (positive = better), and in how many of the 8 years the treated version had the lower MAE.

In [ ]:
PAIRS = {**{w: base for w, base in WEIGHTED.items() if not base.endswith('Huber')},
         'Huber': 'Ridge (all features)', 'Top-k + Huber': 'Top-k + Ridge',
         'Huber, weighted': 'Huber', 'Top-k + Huber, weighted': 'Top-k + Huber'}
pd.DataFrame([{'model': m, 'compared_with': base,
               'MAE_change': overall.loc[m, 'MAE'] - overall.loc[base, 'MAE'],
               'R2_oos_change': overall.loc[m, 'R2_oos'] - overall.loc[base, 'R2_oos'],
               'years_better': int((by_year[m] < by_year[base]).sum())}
              for m, base in PAIRS.items()]).set_index('model').round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
top_models = overall.drop(index=BASELINES).index[:6]
for m in top_models:
    ax.plot(gain.index, gain[m], marker='o', ms=4, ls='--' if m in WEIGHTED else '-', label=m)
ax.axhline(0, color='#333', lw=1)
ax.set_xlabel('forecast year')
ax.set_ylabel('MAE gain over training mean (points)')
ax.set_title('Rolling-origin forecasts, 6 best models (dashed = weighted): above 0 = better than the training mean')
ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

## A5. What each model chose at each origin

Settings picked by the inner CV at each forecast year, and the features top-k kept. Stable choices across origins are a good sign; settings that jump around mean the CV can't tell them apart.

In [ ]:
for name in overall.drop(index=BASELINES).index:
    settings = chosen[chosen['model'] == name].set_index('forecast_year').drop(columns='model').dropna(axis=1, how='all')
    print(f'--- {name} ---')
    display(settings.round(4))

TOPK = [m for m in MODELS if m.startswith('Top-k')]
pd.DataFrame({m: {t: ', '.join(fitted[(t, m)][:-1].get_feature_names_out()) for t in ORIGINS} for m in TOPK})

## A6. What the random forest relies on

Permutation importance at each origin: shuffle one input column in the forecast year's rows and measure how much the MAE gets worse (10 shuffles), averaged over the 8 origins. Computed on the pipeline's input columns, so `ecoregion` counts as one feature. Positive = the model uses it; around 0 = it doesn't.

In [ ]:
imps = []
for t in ORIGINS:
    now = data[data['survey_year'] == t]
    r = permutation_importance(fitted[(t, 'Random forest')], now[features], now[TARGET],
                               scoring=SCORING, n_repeats=10, random_state=SEED)
    imps.append(r.importances_mean)
imps = pd.DataFrame(imps, index=ORIGINS, columns=features)
perm = pd.DataFrame({'mae_increase': imps.mean(), 'se': imps.std(ddof=1) / np.sqrt(len(ORIGINS))})
perm = perm.sort_values('mae_increase', ascending=False)

top = perm.head(15)[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top.index, top['mae_increase'], xerr=top['se'], color='#2e8b57', capsize=2)
ax.axvline(0, color='#777', lw=1)
ax.set_xlabel('increase in forecast MAE when shuffled (points), ±1 SE across origins')
ax.set_title('Random forest: permutation importance (top 15)')
plt.tight_layout()
plt.show()
perm.head(15).round(3)

## A7. Final model: refit on all 2013–2025 data

The model with the lowest rolling-origin MAE (baselines excluded) is tuned once more on all 348 rows with the same year-ordered CV (validating on 2018–2025, each trained on the years before it), refitted on all rows, and saved. To forecast a new year, it needs that year's survey-year heat stress (`noaa_max_dhw`, `noaa_mean_ssta`), i.e. a sea-temperature forecast, plus the previous survey's measurements.

In [ ]:
best_name = overall.drop(index=BASELINES)['MAE'].idxmin()
final_cv = time_cv(data)
final_search = fit_search(best_name, final_cv, data, noise_weights(data)[0])
final = final_search.best_estimator_
print(f'final model: {best_name}, settings {final_search.best_params_}')
print(f'rolling-origin MAE {overall.loc[best_name, "MAE"]:.3f} vs training mean {overall.loc["Training mean", "MAE"]:.3f}')

names = final[:-1].get_feature_names_out()
est = final[-1]
if hasattr(est, 'coef_'):
    weights = pd.Series(est.coef_, index=names, name='coefficient (per SD; flags/one-hot per 0->1)')
else:
    weights = pd.Series(est.feature_importances_, index=names, name='impurity importance')
display(weights[weights != 0].sort_values(key=abs, ascending=False).head(15).round(3).to_frame())

joblib.dump(final, MODEL_DIR / 'lcc_forecast_model.joblib')
reloaded = joblib.load(MODEL_DIR / 'lcc_forecast_model.joblib')
assert np.allclose(reloaded.predict(data[features]), final.predict(data[features]))
print(f'saved {MODEL_DIR / "lcc_forecast_model.joblib"}')

## A8. Tuning check: year-ordered vs island-grouped

Why tune with year-ordered folds? For the final model's type, this repeats the rolling origin with island-grouped tuning (5 folds by island, every fold containing every past year) and compares:
1. **Forecast accuracy** of each tuning method across 2018–2025.
2. **The final model** each method produces on all 348 rows: its settings and how many features it actually uses.

Island-grouped folds share years between training and validation, so a feature that tracks year-wide shocks (such as survey-year heat stress) looks more useful than it is for forecasting a new year, which pushes tuning toward weaker regularisation and more features.

In [ ]:
def island_cv(rows):
    return list(GroupKFold(n_splits=N_FOLDS).split(rows, groups=rows['island']))


def summarize(pred):
    y, ref, yr = preds[TARGET].to_numpy(), preds['Training mean'].to_numpy(), preds['survey_year'].to_numpy()
    beat = sum(mean_absolute_error(y[yr == t], pred[yr == t]) < mean_absolute_error(y[yr == t], ref[yr == t])
               for t in ORIGINS)
    return {'MAE': mean_absolute_error(y, pred), 'R2_oos': 1 - ((y - pred) ** 2).sum() / ((y - ref) ** 2).sum(),
            'years_beating_mean': beat}


def features_used(search):
    est = search.best_estimator_
    names = est[:-1].get_feature_names_out()
    return int((est[-1].coef_ != 0).sum()) if hasattr(est[-1], 'coef_') else len(names)


island_pred = []
for t in ORIGINS:
    past = data[data['survey_year'] < t].reset_index(drop=True)
    now = data[data['survey_year'] == t]
    island_pred.append(fit_search(best_name, island_cv(past), past, noise_weights(past)[0]).predict(now[features]))
island_pred = np.concatenate(island_pred)   # same row order as preds

island_final = fit_search(best_name, island_cv(data), data, noise_weights(data)[0])
print(f'{best_name}: rolling-origin accuracy and final model under each tuning method')
pd.DataFrame({
    'year-ordered tuning (used)': {**summarize(preds[best_name].to_numpy()),
                                   'final_settings': final_search.best_params_, 'final_features_used': features_used(final_search)},
    'island-grouped tuning': {**summarize(island_pred),
                              'final_settings': island_final.best_params_, 'final_features_used': features_used(island_final)},
}).T

# Part B: Do pollution / disturbance predict coral change?

**Data:** all 348 surveys with a target (this is estimation, not a prediction score). Rows missing any model variable are dropped rather than imputed, since imputed values would make the intervals look more certain than they are.

**Model:** ordinary least squares on `lcc_change`, in natural units:

| Term | Meaning |
|---|---|
| `prev_share_{other,sand,disturb,pollution}` | previous survey's share of the non-coral space (%). Reference: available substrate, so each coefficient means "1 point of non-coral space as X instead of available substrate" |
| `lcc_prev` | previous cover (captures regression to the mean) |
| `dhw_peak_prev_year`, `noaa_max_dhw`, `noaa_mean_ssta` | heat stress, previous and survey year |
| `prev_impact_*` | impacts recorded at the previous survey (0/1) |
| `year_c` | year trend (centred) |
| `eco_*` | ecoregion (reference: Sunda Shelf) |

**Uncertainty:** 95% intervals from an island-cluster bootstrap. Whole islands are resampled 2000 times, because an island's surveys aren't independent of each other.

In [ ]:
SHARES = ['prev_share_other', 'prev_share_sand', 'prev_share_disturb', 'prev_share_pollution']
HEAT = ['dhw_peak_prev_year', 'noaa_max_dhw', 'noaa_mean_ssta']
IMPACTS = ['prev_impact_anchor', 'prev_impact_nets', 'prev_impact_trash', 'prev_impact_bleaching', 'prev_impact_cot']

eff = data.copy()
eff['year_c'] = eff['survey_year'] - eff['survey_year'].mean()
eco = pd.get_dummies(eff['ecoregion'], prefix='eco', dtype=float).drop(columns='eco_Sunda Shelf')
EFFECT_COLS = SHARES + ['lcc_prev'] + HEAT + IMPACTS + ['year_c'] + list(eco.columns)

eff = pd.concat([eff, eco], axis=1)
d = eff.dropna(subset=EFFECT_COLS + [TARGET]).reset_index(drop=True)
print(f'{len(d)} of {len(eff)} rows complete ({len(eff) - len(d)} dropped), {d["island"].nunique()} islands')


def ols(X, y):
    X1 = np.column_stack([np.ones(len(X)), X])
    return np.linalg.lstsq(X1, y, rcond=None)[0][1:]


def cluster_bootstrap(data, cols, n_boot=2000, seed=SEED):
    X, y = data[cols].to_numpy(float), data[TARGET].to_numpy(float)
    groups = data['island'].to_numpy()
    islands = np.unique(groups)
    rows_of = {i: np.flatnonzero(groups == i) for i in islands}
    rng = np.random.default_rng(seed)
    draws = np.empty((n_boot, len(cols)))
    for b in range(n_boot):
        rows = np.concatenate([rows_of[i] for i in rng.choice(islands, len(islands))])
        draws[b] = ols(X[rows], y[rows])
    return ols(X, y), draws


def effect_table(coef, draws, cols):
    lo, hi = np.percentile(draws, [2.5, 97.5], axis=0)
    opposite = np.where(coef >= 0, (draws < 0).mean(axis=0), (draws > 0).mean(axis=0))
    return pd.DataFrame({'coef': coef, 'ci_low': lo, 'ci_high': hi,
                         'excludes_0': (lo > 0) | (hi < 0), 'share_opposite_sign': opposite}, index=cols)


coef, draws = cluster_bootstrap(d, EFFECT_COLS)
assert np.array_equal(draws, cluster_bootstrap(d, EFFECT_COLS)[1])   # reproducible with the fixed seed
effects = effect_table(coef, draws, EFFECT_COLS)
effects.round(3)

### Substrate shares per 10 points of non-coral space

E.g. a coefficient of −0.5 means: if 10 more points of last survey's non-coral space were pollution indicators (instead of available substrate), coral cover changed by 0.5 points more negatively this survey.

In [ ]:
per10 = effects.loc[SHARES, ['coef', 'ci_low', 'ci_high']] * 10
per10['excludes_0'] = effects.loc[SHARES, 'excludes_0']
per10.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
panels = [(axes[0], SHARES, [10] * 4, 'Substrate share (per 10 points of non-coral space)'),
          (axes[1], HEAT, [1, 1, 0.1], 'Heat stress (DHW per 1 °C-week; SSTA per 0.1 °C)')]
for ax, cols, scale, title in panels:
    t = effects.loc[cols, ['coef', 'ci_low', 'ci_high']].mul(scale, axis=0)
    ypos = np.arange(len(cols))[::-1]
    for y0, (col, r) in zip(ypos, t.iterrows()):
        color = '#c44e52' if effects.loc[col, 'excludes_0'] else '#2a6f97'
        ax.errorbar(r['coef'], y0, xerr=[[r['coef'] - r['ci_low']], [r['ci_high'] - r['coef']]],
                    fmt='o', color=color, capsize=3)
    ax.axvline(0, color='#777', lw=1)
    ax.set_yticks(ypos, [c.replace('prev_share_', '').replace('_', ' ') for c in cols])
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('effect on lcc_change (points), 95% CI')
fig.suptitle('Red = interval excludes 0', fontsize=9, x=0.99, ha='right')
plt.tight_layout()
plt.show()

### Robustness: one-year changes only

18 targets span a gap of 2–5 years between surveys. Refit on the one-year changes only, to check that those longer gaps aren't driving the share estimates.

In [ ]:
d1 = d[d['years_since_prev'] == 1].reset_index(drop=True)
coef1, draws1 = cluster_bootstrap(d1, EFFECT_COLS)
one_year = effect_table(coef1, draws1, EFFECT_COLS)

compare = pd.concat({f'all rows (n={len(d)})': effects.loc[SHARES + HEAT, ['coef', 'ci_low', 'ci_high']],
                     f'1-year gaps only (n={len(d1)})': one_year.loc[SHARES + HEAT, ['coef', 'ci_low', 'ci_high']]},
                    axis=1)
compare.round(3)

## How to read Part B

- **Time order, not proof of cause.** Predictors come from the *previous* survey, so they precede the change, but this is observational data: something else could drive both (e.g. a coastal development causing sediment and coral loss).
- **"Not detected" ≠ "no effect".** About 50 islands with noisy year-to-year changes can only detect fairly large effects. An interval spanning 0 means the data can't tell, not that the effect is zero.
- **Shares are relative.** Each substrate coefficient compares that category against available substrate, not against "nothing".
- **Survey-year heat stress** (`noaa_max_dhw`, `noaa_mean_ssta`) covers the whole calendar year, so part of it may come after the survey date.